# Senate meeting minutes extractor
Last updated: _2026.10.02_

In [243]:
import glob
import json
import pandas as pd
from pathlib import Path

## English

In [244]:
# English
directory = Path("input")
pattern = "senate_meeting_minutes_hun_part*en-US.json"

# Find all matching JSON files
json_files = sorted(directory.glob(pattern))

if not json_files:
    raise FileNotFoundError(f"No files found matching pattern: {pattern}")

print(f"Found {len(json_files)} files:")
for f in json_files:
    print(f"  - {f.name}")

Found 5 files:
  - senate_meeting_minutes_hun_part1_202610011542 en-US.json
  - senate_meeting_minutes_hun_part2_202610011542 en-US.json
  - senate_meeting_minutes_hun_part3_202610011542 en-US.json
  - senate_meeting_minutes_hun_part4_202610011542 en-US.json
  - senate_meeting_minutes_hun_part5_202610011542 en-US.json


In [245]:
dfs = []
for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # Convert to DataFrame (handles list of records or single dict)
    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict):
        # If the JSON is a dict of lists / nested, try to normalize
        df = pd.json_normalize(data)
    else:
        raise ValueError(f"Unexpected JSON structure in {file_path}")
    
    # Optional: keep track of the source file
    df["source_file"] = file_path.name
    dfs.append(df)

In [246]:
pdf_eng = pd.concat(dfs, ignore_index=True)

print(f"Merged DataFrame shape: {pdf_eng.shape}")

Merged DataFrame shape: (58, 4)


## Hungarian

In [247]:
directory = Path("input")
pattern = "senate_meeting_minutes_hun_part*.json"

# Find all matching JSON files, excluding en-US files
json_files = sorted(
    f for f in directory.glob(pattern)
    if "en-US" not in f.stem
)

if not json_files:
    raise FileNotFoundError(f"No files found matching pattern: {pattern}")

print(f"Found {len(json_files)} files:")
for f in json_files:
    print(f"  - {f.name}")

Found 5 files:
  - senate_meeting_minutes_hun_part1_202610011542.json
  - senate_meeting_minutes_hun_part2_202610011542.json
  - senate_meeting_minutes_hun_part3_202610011542.json
  - senate_meeting_minutes_hun_part4_202610011542.json
  - senate_meeting_minutes_hun_part5_202610011542.json


In [248]:
dfs = []
for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # Convert to DataFrame (handles list of records or single dict)
    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict):
        # If the JSON is a dict of lists / nested, try to normalize
        df = pd.json_normalize(data)
    else:
        raise ValueError(f"Unexpected JSON structure in {file_path}")
    
    # Optional: keep track of the source file
    df["source_file"] = file_path.name
    dfs.append(df)

In [249]:
pdf_hun = pd.concat(dfs, ignore_index=True)

print(f"Merged DataFrame shape: {pdf_hun.shape}")

Merged DataFrame shape: (58, 4)


## Concatenating dataframes

In [251]:
df_all = pd.concat([pdf_hun, pdf_eng], axis=1)
df_all.drop(columns=['year', 'source_file'], inplace=True)
df_all.columns = ['filename', 'text_hun', 'filename_eng', 'text_eng']
df_all.drop(columns=['filename_eng'], inplace=True)

In [252]:
df_all[['filename', 'text_hun', 'text_eng']][:5]

,filename,text_hun,text_eng
0,jegyzokonyv-260603.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...
1,jegyzokonyv-260409.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...
2,jegyzokonyv-260330.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...
3,sz-jegyzokonyv-250317.pdf,————— \n1 | 10 \n \nBruno van Pottelsberghe: J...,————— \n1 | 10 \n \nBruno van Pottelsberghe: G...
4,sz-jegyzokonyv-260303-ai.pdf,MINUTES\ntaken on the 3 March 2026 session of ...,"MINUTES\nof the March 3, 2026, session of the\..."


In [253]:
pdf_summary = pd.read_csv("input/senate_meeting_minutes_summary_202610020950.csv")
pdf_summary.drop(columns=['Unnamed: 0', 'sidebar_label', 'label', 'status', 'error'], inplace=True)

In [254]:
pdf_summary[:5]

,sidebar_url,url,filename
0,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/d4wm.15x...,jegyzokonyv-260603.pdf
1,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cwdy.1ru...,jegyzokonyv-260409.pdf
2,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cv8x.ap3...,jegyzokonyv-260330.pdf
3,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cxp1.bu1...,sz-jegyzokonyv-250317.pdf
4,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/dfvm.13b...,sz-jegyzokonyv-260303-ai.pdf


In [255]:
df_all[:5]

,filename,text_hun,text_eng
0,jegyzokonyv-260603.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...
1,jegyzokonyv-260409.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...
2,jegyzokonyv-260330.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...
3,sz-jegyzokonyv-250317.pdf,————— \n1 | 10 \n \nBruno van Pottelsberghe: J...,————— \n1 | 10 \n \nBruno van Pottelsberghe: G...
4,sz-jegyzokonyv-260303-ai.pdf,MINUTES\ntaken on the 3 March 2026 session of ...,"MINUTES\nof the March 3, 2026, session of the\..."


In [256]:
pdf_all = pd.merge(pdf_summary, df_all, on='filename', how='left')

In [257]:
pdf_dates = pd.read_csv('input/senate_meeting_minutes_summary_202610031820.csv')
pdf_dates = pdf_dates[['date', 'filename']].copy()

In [259]:
pdf_all = pd.merge(pdf_all, pdf_dates, on='filename', how='left')

In [260]:
pdf_all[:5]

,sidebar_url,url,filename,text_hun,text_eng,date
0,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/d4wm.15x...,jegyzokonyv-260603.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...,2026-06-03
1,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cwdy.1ru...,jegyzokonyv-260409.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...,2026-04-09
2,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cv8x.ap3...,jegyzokonyv-260330.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...,2026-03-30
3,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cxp1.bu1...,sz-jegyzokonyv-250317.pdf,————— \n1 | 10 \n \nBruno van Pottelsberghe: J...,————— \n1 | 10 \n \nBruno van Pottelsberghe: G...,2026-03-17
4,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/dfvm.13b...,sz-jegyzokonyv-260303-ai.pdf,MINUTES\ntaken on the 3 March 2026 session of ...,"MINUTES\nof the March 3, 2026, session of the\...",2026-03-03


In [261]:
# cross-checking the output
print("Number of rows with missing English text: {:,}\nNumber of rows with missing Hungarian text: {:,}".format(
    pdf_all['text_hun'].isna().sum(),
    pdf_all['text_eng'].isna().sum()
))

Number of rows with missing English text: 1
Number of rows with missing Hungarian text: 1


In [262]:
from Helper import Log

In [263]:
with Log.save_file("input/senate_meeting_minutes_corpus.csv") as filename:
    pdf_all.to_csv(filename, index=False)

File saved to `input\senate_meeting_minutes_corpus_202610031826.csv` (7.5MB)


## List models

In [43]:
import os
import time
import pandas as pd
from anthropic import Anthropic
from tqdm import tqdm

In [62]:
with open('anthropic.key', 'r') as file:
    ANTHROPIC_API_KEY = file.read().rstrip()

In [73]:
client = Anthropic(api_key=ANTHROPIC_API_KEY)
pages = client.models.list()
page = pages.data[0]
print(page.id)

claude-sonnet-5-5


## Conversation detection

In [74]:
client = Anthropic(api_key=ANTHROPIC_API_KEY)

In [90]:
MODEL = "claude-sonnet-5-5"
MAX_ROWS = None

In [91]:
def ask_claude_if_discussion(text: str) -> dict:
    """
    Sends a single request to Claude asking whether the text contains
    a discussion / conversation.
    Returns a dictionary with the raw answer and a boolean flag.
    """
    if not isinstance(text, str) or not text.strip():
        return {
            "is_discussion": False,
            "raw_answer": "Empty or non-string text",
            "error": None
        }

    prompt = f"""Does the following text contain a discussion or a conversation?

Answer with exactly one of these two options on the first line:
YES
NO

Then, on the next line, give a short one-sentence explanation.

Text:
\"\"\"
{text[:15000]}   # truncate very long texts to stay within context limits
\"\"\"
"""

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=150,
            #temperature=0,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        answer = response.content[0].text.strip()

        # Simple parsing
        first_line = answer.split("\n")[0].strip().upper()
        is_discussion = first_line.startswith("YES")

        return {
            "is_discussion": is_discussion,
            "raw_answer": answer,
            "error": None
        }

    except Exception as e:
        return {
            "is_discussion": None,
            "raw_answer": None,
            "error": str(e)
        }

In [92]:
df = pdf_all.copy()
if MAX_ROWS is not None:
    df = df.head(MAX_ROWS)

results = []

print(f"Processing {len(df)} rows with model {MODEL}...")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    result = ask_claude_if_discussion(row['text_eng'])

    results.append({
        "index": idx,
        "is_discussion": result["is_discussion"],
        "claude_raw_answer": result["raw_answer"],
        "error": result["error"]
    })

Processing 59 rows with model claude-sonnet-5-5...


100%|██████████| 59/59 [01:36<00:00,  1.64s/it]


In [93]:
results_df = pd.DataFrame(results).set_index("index")

df = df.join(results_df)

# Quick summary
print("\n=== Summary ===")
print(df["is_discussion"].value_counts(dropna=False))
print(f"\nErrors: {df['error'].notna().sum()}")


=== Summary ===
is_discussion
True     42
False    17
Name: count, dtype: int64

Errors: 0


In [224]:
df[:5]

,sidebar_url,url,filename,text_hun,text_eng
0,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cxp1.bu1...,sz-jegyzokonyv-250317.pdf,————— \n1 | 10 \n \nBruno van Pottelsberghe: J...,————— \n1 | 10 \n \nBruno van Pottelsberghe: G...
1,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/dfvm.13b...,sz-jegyzokonyv-260303-ai.pdf,MINUTES\ntaken on the 3 March 2026 session of ...,"MINUTES\nof the March 3, 2026, session of the\..."
2,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cxp7.1ew...,sz-jegyzokonyv-260127.pdf,MINUTES \nBruno van Pottelsberghe: Good aftern...,MINUTES \nBruno van Pottelsberghe: Good aftern...
3,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cgxq.1mt...,sz-jegyzokonyv-251211-c.pdf,————— \n1 | 24 \n \nMINUTES \ntaken on the 11 ...,————— \n1 | 24 \n \nMINUTES \nof the December ...
4,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cgy5.11r...,sz-jegyzokonyv-251118-c.pdf,Bruno van Pottelsberghe: Good afternoon everyb...,"Bruno van Pottelsberghe: Good afternoon, every..."


In [97]:
with Log.save_file("input/senate_meeting_minutes_discussions.csv") as filename:
    df[df['is_discussion'] == True].to_csv(filename, index=False)with Log.save_file("input/senate_meeting_minutes_discussions.csv") as filename:
    df[df['is_discussion'] == True].to_csv(filename, index=False)

File saved to `input\senate_meeting_minutes_discussions_202610021125.csv` (7.4MB)


## Conversation processing

In [178]:
import re
import json
from typing import Any, Dict, Optional

In [179]:
df = pd.read_csv('input/senate_meeting_minutes_discussions_202610021125.csv')
df.drop(columns=['Unnamed: 0', 'is_discussion', 'claude_raw_answer', 'error'], inplace=True)

In [180]:
df.shape, df.columns

((42, 5),
 Index(['sidebar_url', 'url', 'filename', 'text_hun', 'text_eng'], dtype='str'))

In [181]:
df[:5]

,sidebar_url,url,filename,text_hun,text_eng
0,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cxp1.bu1...,sz-jegyzokonyv-250317.pdf,————— \n1 | 10 \n \nBruno van Pottelsberghe: J...,————— \n1 | 10 \n \nBruno van Pottelsberghe: G...
1,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/dfvm.13b...,sz-jegyzokonyv-260303-ai.pdf,MINUTES\ntaken on the 3 March 2026 session of ...,"MINUTES\nof the March 3, 2026, session of the\..."
2,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cxp7.1ew...,sz-jegyzokonyv-260127.pdf,MINUTES \nBruno van Pottelsberghe: Good aftern...,MINUTES \nBruno van Pottelsberghe: Good aftern...
3,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cgxq.1mt...,sz-jegyzokonyv-251211-c.pdf,————— \n1 | 24 \n \nMINUTES \ntaken on the 11 ...,————— \n1 | 24 \n \nMINUTES \nof the December ...
4,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cgy5.11r...,sz-jegyzokonyv-251118-c.pdf,Bruno van Pottelsberghe: Good afternoon everyb...,"Bruno van Pottelsberghe: Good afternoon, every..."


In [182]:
with open('prompt/institutional_meeting_transcript_analysis_prompt.md', 'r') as file:
    main_prompt = file.read().rstrip()

In [185]:
def ask_claude(
    prompt_template: str,
    text: str,
    *,
    max_tokens: int = 8192,
    model: str = MODEL,
    system: Optional[str] = None,
    temperature: float = 0.0,
) -> Dict[str, Any]:
    """
    Robust wrapper around Claude that always returns a structured dict.
    Never raises on JSON parsing problems – returns success=False instead.
    """
    if not isinstance(text, str) or not text.strip():
        return {
            "success": False,
            "data": None,
            "raw_answer": None,
            "error": "Empty or non-string text",
        }

    if system is None:
        system = (
            "You are a helpful assistant that always responds with valid JSON only. "
            "Do not output any text before or after the JSON object. "
            "Do not use markdown code blocks."
        )

    user_content = prompt_template.format(text)

    try:
        response = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            system=system,
            messages=[{"role": "user", "content": user_content}],
            thinking={"type": "between_tools"}
        )
    except Exception as e:
        return {
            "success": False,
            "data": None,
            "raw_answer": None,
            "error": f"API call failed: {e}",
        }

    # ------------------------------------------------------------------
    # 1. Extract all text blocks (handles ThinkingBlock + TextBlock)
    # ------------------------------------------------------------------
    text_parts = []
    for block in response.content:
        # Standard text block
        if getattr(block, "type", None) == "text" and hasattr(block, "text"):
            text_parts.append(block.text)
        # Fallback for any block that has .text
        elif hasattr(block, "text") and block.text:
            text_parts.append(block.text)
        # Optional: also capture thinking if you want to log it
        elif getattr(block, "type", None) == "thinking" and hasattr(block, "thinking"):
            # log it, but don't use it as the answer
            pass

    raw_answer = "".join(text_parts).strip()
    if not raw_answer:
        return {
            "success": False,
            "data": None,
            "raw_answer": None,
            "error": "No text content found in the response",
        }

    # ------------------------------------------------------------------
    # 2. Multi-stage JSON recovery
    # ------------------------------------------------------------------
    def _try_parse(candidate: str) -> Optional[Any]:
        try:
            return json.loads(candidate)
        except (json.JSONDecodeError, TypeError, ValueError):
            return None

    def _clean_json(s: str) -> str:
        """Aggressive but safe cleaning of common LLM JSON defects."""
        s = s.strip()

        # Remove markdown fences
        if s.startswith("```"):
            s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
            s = re.sub(r"\s*```$", "", s)

        # Remove any leading/trailing non-JSON text (common when model adds comments)
        # Find the outermost { ... } or [ ... ]
        match = re.search(r"(\{.*\}|\[.*\])", s, re.DOTALL)
        if match:
            s = match.group(1)

        # Fix unescaped control characters inside strings (very common)
        # Replace literal newlines/tabs that are not already escaped
        s = re.sub(r'(?<!\\)\n', r'\\n', s)
        s = re.sub(r'(?<!\\)\r', r'\\r', s)
        s = re.sub(r'(?<!\\)\t', r'\\t', s)

        # Remove trailing commas before } or ]
        s = re.sub(r",\s*([}\]])", r"\1", s)

        # Fix single quotes used as string delimiters (rare but happens)
        # Only do this carefully – we leave it to the more advanced repair below

        return s.strip()

    # Stage A – direct parse
    data = _try_parse(raw_answer)
    if data is not None:
        return {"success": True, "data": data, "raw_answer": raw_answer, "error": None}

    # Stage B – basic cleaning
    cleaned = _clean_json(raw_answer)
    data = _try_parse(cleaned)
    if data is not None:
        return {"success": True, "data": data, "raw_answer": raw_answer, "error": None}

    # Stage C – more aggressive repair with json_repair (if available)
    try:
        from json_repair import repair_json
        repaired = repair_json(cleaned, return_objects=False)
        data = _try_parse(repaired)
        if data is not None:
            return {"success": True, "data": data, "raw_answer": raw_answer, "error": None}
    except ImportError:
        pass  # json_repair not installed – continue
    except Exception:
        pass

    # Stage D – last-resort: try to close truncated JSON
    # (common when max_tokens cuts the response mid-string)
    if cleaned.count("{") > cleaned.count("}"):
        candidate = cleaned + "}" * (cleaned.count("{") - cleaned.count("}"))
        data = _try_parse(candidate)
        if data is not None:
            return {
                "success": True,
                "data": data,
                "raw_answer": raw_answer,
                "error": "Recovered from truncated JSON (auto-closed braces)",
            }

    if cleaned.count("[") > cleaned.count("]"):
        candidate = cleaned + "]" * (cleaned.count("[") - cleaned.count("]"))
        data = _try_parse(candidate)
        if data is not None:
            return {
                "success": True,
                "data": data,
                "raw_answer": raw_answer,
                "error": "Recovered from truncated JSON (auto-closed brackets)",
            }

    # Stage E – give up gracefully
    return {
        "success": False,
        "data": None,
        "raw_answer": raw_answer,
        "error": "Failed to parse JSON after multiple recovery attempts",
    }

In [214]:
CHECKPOINT_EVERY = 5
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

CHECKPOINT_FILE = CHECKPOINT_DIR / "claude_results_checkpoint.json"
FINAL_OUTPUT_FILE = CHECKPOINT_DIR / "claude_results_final.json"

# ----------------------------------------------------------------------
# Load existing checkpoint if it exists (resume support)
# ----------------------------------------------------------------------
results = []
start_idx = 0

if CHECKPOINT_FILE.exists():
    print(f"Found existing checkpoint → loading {CHECKPOINT_FILE}")
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        results = json.load(f)
    
    # Determine from which index we should continue
    if results:
        last_processed_index = results[-1]["index"]
        # Find the position in the dataframe
        processed_indices = {r["index"] for r in results}
        remaining_indices = [idx for idx in df.index if idx not in processed_indices]
        
        if remaining_indices:
            start_idx = remaining_indices[0]
            print(f"Resuming from index {start_idx} "
                  f"({len(results)} records already processed)")
        else:
            print("All records already processed according to the checkpoint.")
            start_idx = None
    else:
        start_idx = df.index[0]
else:
    print("No checkpoint found → starting from the beginning")

# ----------------------------------------------------------------------
# Main processing loop with checkpointing
# ----------------------------------------------------------------------
if start_idx is not None:
    # Create an iterator that starts from the correct position
    rows_to_process = df.loc[start_idx:]
    
    print(f"Processing {len(rows_to_process)} remaining rows with model {MODEL}...")
    
    for i, (idx, row) in enumerate(tqdm(rows_to_process.iterrows(), 
                                        total=len(rows_to_process))):
        result = ask_claude(main_prompt, row["text_hun"])
        
        results.append({
            "index": idx,
            "success": result['success'],
            "data": result['data'],
            "raw_answer": result['raw_answer'],
            "error": result['error']
        })

        if not result["success"]:
            print(f"Row {i} failed: {result['error']}")
        
        # --------------------------------------------------------------
        # Checkpoint every N records
        # --------------------------------------------------------------
        if (i + 1) % CHECKPOINT_EVERY == 0:
            with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            print(f"\n✓ Checkpoint saved ({len(results)} records) → {CHECKPOINT_FILE}")

# ----------------------------------------------------------------------
# Final save
# ----------------------------------------------------------------------
with open(FINAL_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✓ Final results saved → {FINAL_OUTPUT_FILE}")
print(f"Total records processed: {len(results)}")

Found existing checkpoint → loading checkpoints\claude_results_checkpoint.json
Resuming from index 26 (26 records already processed)
Processing 16 remaining rows with model claude-sonnet-5-5...


 31%|███▏      | 5/16 [03:57<08:50, 48.23s/it]


✓ Checkpoint saved (31 records) → checkpoints\claude_results_checkpoint.json


 62%|██████▎   | 10/16 [07:54<04:44, 47.43s/it]


✓ Checkpoint saved (36 records) → checkpoints\claude_results_checkpoint.json


 94%|█████████▍| 15/16 [11:50<00:47, 47.79s/it]


✓ Checkpoint saved (41 records) → checkpoints\claude_results_checkpoint.json


100%|██████████| 16/16 [12:36<00:00, 47.29s/it]



✓ Final results saved → checkpoints\claude_results_final.json
Total records processed: 42


In [215]:
results_df = pd.DataFrame(results).set_index("index")

In [216]:
results_df['success'].value_counts()

success
True    42
Name: count, dtype: int64

In [217]:
results_df.shape

(42, 4)

In [218]:
results_df[results_df['success'] == False][:5]

,success,data,raw_answer,error
index,,,,


In [219]:
results_df[:5]

,success,data,raw_answer,error
index,,,,
0,True,"{'contains_conversation': 1, 'contains_voting'...","{""contains_conversation"":1,""contains_voting"":1...",None
1,True,"{'contains_conversation': 1, 'contains_voting'...","{""contains_conversation"":1,""contains_voting"":1...",None
2,True,"{'contains_conversation': 1, 'contains_voting'...","{""contains_conversation"":1,""contains_voting"":1...",None
3,True,"{'contains_conversation': 1, 'contains_voting'...","{""contains_conversation"":1,""contains_voting"":1...",None
4,True,"{'contains_conversation': 1, 'contains_voting'...","{""contains_conversation"":1,""contains_voting"":1...",None


In [220]:
expanded_df = pd.json_normalize(results_df['data'])

In [221]:
expanded_df[:5]

,contains_conversation,contains_voting,discussion_segments,actors,concepts_discussed,attitude_matrix.actors,voting_events
index,,,,,,,
0,1,1,"[{'segment_id': 'S1', 'title': 'Adoption of th...","[{'actor_id': 'A1', 'actor': 'Bruno van Pottel...","[{'concept_id': 'C1', 'concept': 'Code of Ethi...","[{'actor_id': 'A1', 'actor': 'Bruno van Pottel...",NaN
1,1,1,"[{'segment_id': 'S1', 'title': 'Agenda adoptio...","[{'actor_id': 'A1', 'actor': 'Judit Nagy'}, {'...","[{'concept_id': 'C1', 'concept': 'Postponement...","[{'actor_id': 'A1', 'actor': 'Judit Nagy', 'at...",NaN
2,1,1,"[{'segment_id': 'S1', 'title': 'Amendment of t...","[{'actor_id': 'A1', 'actor': 'Bruno van Pottel...","[{'concept_id': 'C1', 'concept': 'Amendment of...","[{'actor_id': 'A1', 'actor': 'Bruno van Pottel...","[{'vote_id': 'V1', 'subject': 'Test vote', 'se..."
3,1,1,"[{'segment_id': 'S1', 'title': 'Opening, agend...","[{'actor_id': 'A1', 'actor': 'Bruno van Pottel...","[{'concept_id': 'C1', 'concept': 'Verbatim rec...",NaN,NaN
4,1,1,"[{'segment_id': 'S1', 'title': 'Opening, quoru...","[{'actor_id': 'A1', 'actor': 'Bruno van Pottel...","[{'concept_id': 'C1', 'concept': 'Procedural a...","[{'actor_id': 'A1', 'actor': 'Bruno van Pottel...",NaN


In [267]:
pdf_all = pd.read_csv('input/senate_meeting_minutes_corpus_202610031826.csv')
pdf_all.drop(columns=['Unnamed: 0'], inplace=True)

In [268]:
pdf_all[:5]

,sidebar_url,url,filename,text_hun,text_eng,date
0,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/d4wm.15x...,jegyzokonyv-260603.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...,2026-06-03
1,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cwdy.1ru...,jegyzokonyv-260409.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...,2026-04-09
2,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cv8x.ap3...,jegyzokonyv-260330.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...,2026-03-30
3,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cxp1.bu1...,sz-jegyzokonyv-250317.pdf,————— \n1 | 10 \n \nBruno van Pottelsberghe: J...,————— \n1 | 10 \n \nBruno van Pottelsberghe: G...,2026-03-17
4,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/dfvm.13b...,sz-jegyzokonyv-260303-ai.pdf,MINUTES\ntaken on the 3 March 2026 session of ...,"MINUTES\nof the March 3, 2026, session of the\...",2026-03-03


In [271]:
pdf_all.shape, expanded_df.shape

((59, 6), (42, 7))

In [272]:
# Align expanded_df (42 discussion rows) with pdf_all (59 rows) via filename.
# expanded_df was produced in the same row order as the discussions CSV,
# so we can recover the key without re-running Claude.
discussions_meta = pd.read_csv(
    "input/senate_meeting_minutes_discussions_202610021125.csv",
    usecols=["filename"],
)

if len(discussions_meta) != len(expanded_df):
    raise ValueError(
        f"Row count mismatch: discussions CSV has {len(discussions_meta)} rows, "
        f"expanded_df has {len(expanded_df)}. Cannot align safely."
    )

expanded_with_key = expanded_df.copy()
expanded_with_key["filename"] = discussions_meta["filename"].values

result = pdf_all.merge(
    expanded_with_key[
        [
            "filename",
            "discussion_segments",
            "actors",
            "concepts_discussed",
            "attitude_matrix.actors",
            "voting_events",
        ]
    ],
    on="filename",
    how="left",
)

print(f"result shape: {result.shape}")
print(f"Rows with discussion data: {result['discussion_segments'].notna().sum()}")
print(f"Rows without discussion data: {result['discussion_segments'].isna().sum()}")
result.head()

result shape: (59, 11)
Rows with discussion data: 42
Rows without discussion data: 17


,sidebar_url,url,filename,text_hun,text_eng,date,discussion_segments,actors,concepts_discussed,attitude_matrix.actors,voting_events
0,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/d4wm.15x...,jegyzokonyv-260603.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...,2026-06-03,NaN,NaN,NaN,NaN,NaN
1,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cwdy.1ru...,jegyzokonyv-260409.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...,2026-04-09,NaN,NaN,NaN,NaN,NaN
2,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cv8x.ap3...,jegyzokonyv-260330.pdf,SZENÁTUS \nÜlésen kívüli szavazás \nküldi: \nJ...,SENATE \nVoting Outside of a Meeting \nSent by...,2026-03-30,NaN,NaN,NaN,NaN,NaN
3,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/cxp1.bu1...,sz-jegyzokonyv-250317.pdf,————— \n1 | 10 \n \nBruno van Pottelsberghe: J...,————— \n1 | 10 \n \nBruno van Pottelsberghe: G...,2026-03-17,"[{'segment_id': 'S1', 'title': 'Adoption of th...","[{'actor_id': 'A1', 'actor': 'Bruno van Pottel...","[{'concept_id': 'C1', 'concept': 'Code of Ethi...","[{'actor_id': 'A1', 'actor': 'Bruno van Pottel...",NaN
4,https://www.uni-corvinus.hu/fooldal/egyetemunk...,https://www.uni-corvinus.hu/downloads/dfvm.13b...,sz-jegyzokonyv-260303-ai.pdf,MINUTES\ntaken on the 3 March 2026 session of ...,"MINUTES\nof the March 3, 2026, session of the\...",2026-03-03,"[{'segment_id': 'S1', 'title': 'Agenda adoptio...","[{'actor_id': 'A1', 'actor': 'Judit Nagy'}, {'...","[{'concept_id': 'C1', 'concept': 'Postponement...","[{'actor_id': 'A1', 'actor': 'Judit Nagy', 'at...",NaN


In [275]:
result[result['discussion_segments'].isna() == False].shape

(42, 11)

In [276]:
with Log.save_file("input/senate_meeting_minutes_extracted_discussions.csv") as filename:
    result[result['discussion_segments'].isna() == False].to_csv(filename, index=False)

File saved to `input\senate_meeting_minutes_extracted_discussions_202610031847.csv` (8.2MB)


## Conversion to MaxQDA-compatible format

In [278]:
# ============================================================
# Conversion to Leifeld / DNA format (MaxQDA-compatible)
# One CSV per meeting / row
# ============================================================
import json
import ast
import pandas as pd
from pathlib import Path
from typing import Any, Dict, List, Optional, Union
from datetime import datetime

# ------------------------------------------------------------------
# Helpers (unchanged)
# ------------------------------------------------------------------
def _safe_load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def _parse_maybe_json(value: Any) -> Any:
    """Turn a string that looks like a Python/JSON literal into an object."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, (list, dict)):
        return value
    if isinstance(value, str):
        value = value.strip()
        if not value or value.lower() in {"nan", "none", "null"}:
            return None
        try:
            return json.loads(value)
        except Exception:
            try:
                return ast.literal_eval(value)
            except Exception:
                return value
    return value

def _attitude_to_agreement(attitude: Any) -> Optional[int]:
    if attitude is None:
        return None
    mapping = {
        "positive": 1,
        "support": 1,
        "favour": 1,
        "favor": 1,
        "agree": 1,
        "negative": -1,
        "oppose": -1,
        "against": -1,
        "disagree": -1,
        "neutral": 0,
        "ambivalent": 0,
        "mixed": 0,
    }
    return mapping.get(str(attitude).strip().lower())

def _extract_statements_from_meeting(
    meeting: Dict[str, Any],
    document_id: str,
    date: str = "",
    meeting_index: Optional[int] = None,
) -> List[Dict[str, Any]]:
    """
    Turn one Claude 'data' object into a list of statement dicts.
    Never raises on missing / None nested structures.
    """
    statements: List[Dict[str, Any]] = []
    statement_counter = 1

    actors_list = meeting.get("actors") or []
    concepts_list = meeting.get("concepts_discussed") or []

    if not isinstance(actors_list, list):
        actors_list = []
    if not isinstance(concepts_list, list):
        concepts_list = []

    actors = {
        a.get("actor_id"): a.get("actor") or a.get("name") or a.get("actor_id")
        for a in actors_list
        if isinstance(a, dict) and a.get("actor_id")
    }

    concepts = {
        c.get("concept_id"): c.get("concept") or c.get("name") or c.get("concept_id")
        for c in concepts_list
        if isinstance(c, dict) and c.get("concept_id")
    }

    # Primary path: actor_attitudes nested inside each concept
    for concept_data in concepts_list:
        if not isinstance(concept_data, dict):
            continue
        concept_id = concept_data.get("concept_id")
        concept_name = concept_data.get("concept") or concepts.get(concept_id, concept_id)
        actor_attitudes = concept_data.get("actor_attitudes") or []
        if not isinstance(actor_attitudes, list):
            continue

        for att in actor_attitudes:
            if not isinstance(att, dict):
                continue
            actor_id = att.get("actor_id")
            attitude = att.get("attitude")
            if attitude in (None, "not_discussed", "not discussed"):
                continue
            agreement = _attitude_to_agreement(attitude)
            if agreement is None:
                continue

            actor_name = actors.get(actor_id, actor_id)
            segment_ids = (
                att.get("discussion_segment_ids")
                or att.get("segment_ids")
                or [""]
            )
            if not isinstance(segment_ids, list) or not segment_ids:
                segment_ids = [""]

            evidence = att.get("evidence") or att.get("quote") or ""
            confidence = att.get("confidence") or ""
            statement_type = (
                "procedural/informational" if agreement == 0 else "substantive"
            )

            for seg_id in segment_ids:
                if meeting_index is not None:
                    stmt_id = f"{document_id}_M{meeting_index:02d}_S{statement_counter:04d}"
                else:
                    stmt_id = f"{document_id}_S{statement_counter:04d}"

                statements.append({
                    "statement_id": stmt_id,
                    "date": date,
                    "document_id": document_id,
                    "meeting_index": meeting_index if meeting_index is not None else "",
                    "segment_id": seg_id,
                    "actor_id": actor_id,
                    "actor": actor_name,
                    "concept_id": concept_id,
                    "concept": concept_name,
                    "agreement": agreement,
                    "statement_type": statement_type,
                    "confidence": confidence,
                    "evidence": evidence,
                })
                statement_counter += 1

    # Fallback: attitude_matrix.actors
    if not statements:
        matrix = meeting.get("attitude_matrix")
        if isinstance(matrix, dict):
            matrix_actors = matrix.get("actors")
            if not isinstance(matrix_actors, list):
                matrix_actors = []

            for actor_entry in matrix_actors:
                if not isinstance(actor_entry, dict):
                    continue
                actor_id = actor_entry.get("actor_id")
                actor_name = (
                    actor_entry.get("actor")
                    or actors.get(actor_id, actor_id)
                )
                atts = (
                    actor_entry.get("attitudes")
                    or actor_entry.get("concepts")
                    or []
                )
                if not isinstance(atts, list):
                    continue

                for att in atts:
                    if not isinstance(att, dict):
                        continue
                    concept_id = att.get("concept_id")
                    attitude = att.get("attitude")
                    agreement = _attitude_to_agreement(attitude)
                    if agreement is None:
                        continue

                    if meeting_index is not None:
                        stmt_id = f"{document_id}_M{meeting_index:02d}_S{statement_counter:04d}"
                    else:
                        stmt_id = f"{document_id}_S{statement_counter:04d}"

                    statements.append({
                        "statement_id": stmt_id,
                        "date": date,
                        "document_id": document_id,
                        "meeting_index": meeting_index if meeting_index is not None else "",
                        "segment_id": "",
                        "actor_id": actor_id,
                        "actor": actor_name,
                        "concept_id": concept_id,
                        "concept": concepts.get(concept_id, concept_id),
                        "agreement": agreement,
                        "statement_type": (
                            "substantive" if agreement != 0 else "procedural/informational"
                        ),
                        "confidence": att.get("confidence", ""),
                        "evidence": att.get("evidence", ""),
                    })
                    statement_counter += 1

    return statements

def _extract_date_from_row(row: pd.Series, default_date: str = "") -> str:
    """
    Try common column names for a meeting date.
    Returns a clean YYYY-MM-DD string or the default.
    """
    candidates = [
        "date", "meeting_date", "session_date", "document_date",
        "Date", "Meeting_Date", "sessionDate"
    ]
    for col in candidates:
        if col in row and pd.notna(row[col]):
            val = str(row[col]).strip()
            if not val:
                continue
            # Try to normalise to YYYY-MM-DD
            for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%Y%m%d", "%d-%m-%Y"):
                try:
                    return datetime.strptime(val[:10], fmt).strftime("%Y-%m-%d")
                except ValueError:
                    continue
            # Fallback: take first 10 characters if it looks date-like
            if len(val) >= 8:
                return val[:10].replace("/", "-").replace(".", "-")
    return default_date or "unknown_date"

# ------------------------------------------------------------------
# New main function: one CSV per row
# ------------------------------------------------------------------
def convert_csv_to_dna_csvs(
    csv_file: Union[str, Path],
    output_folder: Union[str, Path],
    document_id_prefix: str = "senate",
    default_date: str = "",
    filename_suffix: str = "dna",
) -> List[Path]:
    """
    Convert each row of the expanded CSV into its own Leifeld-style
    DNA statement CSV.

    Filename pattern: {date}_{original_stem}_{filename_suffix}.csv
    (or {date}_{original_stem}_{rowindex:04d}_{filename_suffix}.csv
     when several rows share the same date)

    Parameters
    ----------
    csv_file : path to the source CSV
    output_folder : directory where individual CSVs will be written
    document_id_prefix : prefix used inside statement_id / document_id
    default_date : used when a row has no recognisable date column
    filename_suffix : string inserted before the .csv extension

    Returns
    -------
    List of Path objects of the files that were written.
    """
    csv_file = Path(csv_file)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    original_stem = csv_file.stem          # e.g. senate_meeting_minutes_extracted_discussions_202610021546
    raw_df = pd.read_csv(csv_file)

    written_files: List[Path] = []
    date_counts: Dict[str, int] = {}       # to disambiguate identical dates

    for i, row in raw_df.iterrows():
        # ---- build meeting dict (same logic as before) ----
        actors = _parse_maybe_json(row.get("actors"))
        concepts = _parse_maybe_json(row.get("concepts_discussed"))
        segments = _parse_maybe_json(row.get("discussion_segments"))
        matrix_actors = _parse_maybe_json(row.get("attitude_matrix.actors"))
        voting = _parse_maybe_json(row.get("voting_events"))

        meeting: Dict[str, Any] = {}
        if isinstance(actors, list):
            meeting["actors"] = actors
        if isinstance(concepts, list):
            meeting["concepts_discussed"] = concepts
        if isinstance(segments, list):
            meeting["discussion_segments"] = segments
        if isinstance(matrix_actors, list):
            meeting["attitude_matrix"] = {"actors": matrix_actors}
        if isinstance(voting, list):
            meeting["voting_events"] = voting

        # ---- date & document id ----
        date_str = _extract_date_from_row(row, default_date)
        doc_id = f"{document_id_prefix}_{i}"

        stmts = _extract_statements_from_meeting(
            meeting,
            document_id=doc_id,
            date=date_str,
            meeting_index=i,
        )

        if not stmts:
            print(f"  [row {i}] no statements – skipped")
            continue

        df = pd.DataFrame(stmts)

        # ---- unique filename ----
        count = date_counts.get(date_str, 0)
        date_counts[date_str] = count + 1

        if count == 0:
            fname = f"{date_str}_{original_stem}_{filename_suffix}.csv"
        else:
            fname = f"{date_str}_{original_stem}_{i:04d}_{filename_suffix}.csv"

        out_path = output_folder / fname
        df.to_csv(out_path, index=False, encoding="utf-8-sig")
        written_files.append(out_path)
        print(f"✓ row {i:4d} → {out_path.name}  ({len(df)} statements)")

    print(f"\nFinished. Wrote {len(written_files)} files into {output_folder}")
    return written_files

In [279]:
written = convert_csv_to_dna_csvs(
    csv_file="input/senate_meeting_minutes_extracted_discussions_202610031847.csv",
    output_folder="output",          # ← change to any folder you like
    document_id_prefix="senate",
    default_date="",                 # leave empty to force extraction from row
    filename_suffix="dna",
)

✓ row    0 → 2026-03-17_senate_meeting_minutes_extracted_discussions_202610031847_dna.csv  (36 statements)
✓ row    1 → 2026-03-03_senate_meeting_minutes_extracted_discussions_202610031847_dna.csv  (30 statements)
✓ row    2 → 2026-01-27_senate_meeting_minutes_extracted_discussions_202610031847_dna.csv  (15 statements)
✓ row    3 → 2025-12-11_senate_meeting_minutes_extracted_discussions_202610031847_dna.csv  (47 statements)
✓ row    4 → 2025-11-18_senate_meeting_minutes_extracted_discussions_202610031847_dna.csv  (32 statements)
✓ row    5 → 2025-10-14_senate_meeting_minutes_extracted_discussions_202610031847_dna.csv  (48 statements)
✓ row    6 → 2025-09-16_senate_meeting_minutes_extracted_discussions_202610031847_dna.csv  (54 statements)
✓ row    7 → 2025-06-24_senate_meeting_minutes_extracted_discussions_202610031847_dna.csv  (49 statements)
✓ row    8 → 2025-05-20_senate_meeting_minutes_extracted_discussions_202610031847_dna.csv  (27 statements)
✓ row    9 → 2025-04-15_senate_meetin